# Análise dos jogos da Steam

Rode cada célula com **Shift+Enter**.

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)

# O cabeçalho do games.csv junta "Discount" e "DLC count" em uma coluna só,
# então lemos o cabeçalho, separamos essas duas e pulamos a linha original
colunas = pd.read_csv('games.csv', nrows=0).columns.tolist()
i = colunas.index('DiscountDLC count')
colunas[i:i + 1] = ['Discount', 'DLC count']

# Colunas usadas na análise: identificação, gênero/categoria, preço,
# avaliação dos jogadores, popularidade e data de lançamento
usar = ['AppID', 'Name', 'Genres', 'Categories', 'Price',
        'Positive', 'Negative', 'Estimated owners', 'Release date']
df = pd.read_csv('games.csv', names=colunas, header=0, index_col=False, usecols=usar)
df = df[usar]

In [2]:
df.head()

,AppID,Name,Genres,Categories,Price,Positive,Negative,Estimated owners,Release date
0,2539430,Black Dragon Mage Playtest,NaN,NaN,0.00,0,0,0 - 0,"Aug 1, 2023"
1,496350,Supipara - Chapter 1 Spring Has Come!,Adventure,"Single-player,Steam Trading Cards,Steam Cloud,...",5.24,252,3,0 - 20000,"Jul 29, 2016"
2,1034400,Mystery Solitaire The Black Raven,Casual,"Single-player,Family Sharing",4.99,21,3,0 - 20000,"May 6, 2019"
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Casual,Indie,Simulation","Single-player,Steam Achievements,Family Sharing",8.99,0,0,0 - 20000,"Oct 31, 2024"
4,3631080,Maze Quest VR,"Action,Early Access","Single-player,VR Only,Steam Leaderboards,Famil...",4.99,0,0,0 - 20000,"Apr 24, 2025"


In [ ]:
df.info()

In [3]:
df['Release date'] = pd.to_datetime(df['Release date'], format='mixed', errors='coerce')

print(df['Release date'].dtype)
print('Datas que não converteram:', df['Release date'].isna().sum())
df['Release date'].head()

datetime64[us]
Datas que não converteram: 0


0   2023-08-01
1   2016-07-29
2   2019-05-06
3   2024-10-31
4   2025-04-24
Name: Release date, dtype: datetime64[us]

In [4]:
df['Estimated owners'].value_counts()

Estimated owners
0 - 20000                75705
0 - 0                    24579
20000 - 50000            11397
50000 - 100000            5355
100000 - 200000           3454
200000 - 500000           2856
500000 - 1000000          1155
1000000 - 2000000          729
2000000 - 5000000          405
5000000 - 10000000         125
10000000 - 20000000         51
20000000 - 50000000         31
50000000 - 100000000         9
100000000 - 200000000        4
Name: count, dtype: int64

In [5]:
faixas = df['Estimated owners'].str.split(' - ', expand=True).astype(int)

df['Owners min'] = faixas[0]
df['Owners max'] = faixas[1]
df['Owners'] = (df['Owners min'] + df['Owners max']) / 2

df[['Estimated owners', 'Owners min', 'Owners max', 'Owners']].head()

,Estimated owners,Owners min,Owners max,Owners
0,0 - 0,0,0,0.0
1,0 - 20000,0,20000,10000.0
2,0 - 20000,0,20000,10000.0
3,0 - 20000,0,20000,10000.0
4,0 - 20000,0,20000,10000.0


In [6]:
sem_donos = df['Owners max'] == 0
sem_genero = df['Genres'].isna()

print('Sem donos:', sem_donos.sum())
print('Sem gênero:', sem_genero.sum())
print('Sem donos E sem gênero:', (sem_donos & sem_genero).sum())

df[sem_donos][['Name', 'Genres', 'Price', 'Positive', 'Release date']].sample(10)

Sem donos: 24579
Sem gênero: 8423
Sem donos E sem gênero: 8298


,Name,Genres,Price,Positive,Release date
123598,Double Cross Dungeon,"Action,Casual,Indie,Free To Play",0.00,0,2026-01-23
84610,Last Flag Playtest,NaN,0.00,0,2025-07-15
123971,Icebound Secrets 2: Soul Hunter,"Adventure,Casual",6.99,0,2026-01-30
122940,wonderking,"Massively Multiplayer,RPG,Free To Play",0.00,0,2026-01-12
79225,Entwined Time,"Casual,Free To Play",0.00,0,2024-10-22
66324,Hentai Detective,"Casual,Indie",2.99,0,2025-12-04
64235,Space Entity Playtest,NaN,0.00,0,2023-09-30
38724,Reflex King of Elements Playtest,NaN,0.00,0,2022-10-14
25200,No Work No Life,"Adventure,Indie,Free To Play",0.00,0,2024-06-10
47422,Ctrl Alt Deal,"Indie,RPG,Simulation,Strategy",15.99,0,2025-07-22


In [7]:
df = df.dropna(subset=['Genres'])
df['Tem donos'] = df['Owners max'] > 0

print('Jogos restantes:', len(df))
print(df['Tem donos'].value_counts())

Jogos restantes: 117432
Tem donos
True     101151
False     16281
Name: count, dtype: int64


In [8]:
generos = df.assign(Genre=df['Genres'].str.split(',')).explode('Genre')

print('Linhas antes:', len(df))
print('Linhas depois:', len(generos))
print('Gêneros diferentes:', generos['Genre'].nunique())

generos['Genre'].value_counts()

Linhas antes: 117432
Linhas depois: 338583
Gêneros diferentes: 33


Genre
Indie                    82884
Casual                   51677
Action                   47344
Adventure                46293
Simulation               24912
Strategy                 23058
RPG                      21577
Free To Play             12605
Early Access             11572
Sports                    4971
Racing                    4169
Massively Multiplayer     2596
Utilities                 1091
Design & Illustration      584
Animation & Modeling       484
Education                  464
Violent                    414
Video Production           340
Game Development           323
Gore                       257
Audio Production           229
Software Training          222
Photo Editing              183
Web Publishing             113
Nudity                      94
Sexual Content              88
Accounting                  32
Movie                        2
Documentary                  1
Episodic                     1
Short                        1
Tutorial                     1
36

In [ ]:
# Totais por gênero. A % positiva usa a soma das avaliações do gênero,
# então jogos com muitas avaliações pesam mais no resultado
por_genero = generos.groupby('Genre').agg(
    Jogos=('AppID', 'count'),
    Positivas=('Positive', 'sum'),
    Negativas=('Negative', 'sum'),
    Donos=('Owners', 'sum'),
    Donos_mediana=('Owners', 'median'),
)
por_genero['% positivas'] = por_genero['Positivas'] / (por_genero['Positivas'] + por_genero['Negativas']) * 100

por_genero.sort_values('Donos', ascending=False)

In [ ]:
# A soma é dominada pelos grandes sucessos. Para ver o jogo "típico" de cada
# gênero, calculamos a % positiva de cada jogo (só com 10+ avaliações, para
# um jogo com 1 avaliação não contar como 100%) e pegamos a mediana
avaliacoes = generos['Positive'] + generos['Negative']
com_avaliacoes = generos[avaliacoes >= 10].copy()
com_avaliacoes['% positivas'] = com_avaliacoes['Positive'] / (com_avaliacoes['Positive'] + com_avaliacoes['Negative']) * 100

por_genero['% positivas (mediana por jogo)'] = com_avaliacoes.groupby('Genre')['% positivas'].median()
por_genero['Jogos com 10+ avaliações'] = com_avaliacoes.groupby('Genre').size()

# Gêneros com poucos jogos (ex.: 1 jogo com 100%) bagunçam o ranking
por_genero[por_genero['Jogos'] >= 100].sort_values('% positivas', ascending=False)

In [ ]:
# Exporta os dados limpos para o Power BI, em duas tabelas:
# - jogos.csv: um jogo por linha
# - jogos_generos.csv: um par jogo/gênero por linha (liga com jogos.csv pelo AppID)
# Separador ";" e decimal "," para o Power BI em português ler os números certo
colunas_jogos = ['AppID', 'Name', 'Price', 'Positive', 'Negative',
                 'Owners min', 'Owners max', 'Owners', 'Tem donos', 'Release date']
df[colunas_jogos].to_csv('jogos.csv', sep=';', decimal=',', index=False, encoding='utf-8-sig')
generos[['AppID', 'Genre']].to_csv('jogos_generos.csv', sep=';', index=False, encoding='utf-8-sig')

print('jogos.csv:', len(df), 'linhas')
print('jogos_generos.csv:', len(generos), 'linhas')